# 把 Fish Audio Drama 3 原版快照存入 Google Drive

在 Colab 选择 **CPU** 运行时，确保 Drive 至少有 **42 GB** 可用空间，随后按顺序运行两格，并在第一格完成 Google 授权。文件从 Hugging Face 直接传入 Drive。请只在一个会话运行。

中断后重新打开笔记本并运行两格。已校验的文件会跳过；发现同名文件内容不同会停止，请先检查或移走冲突文件。强制终止会话可能留下以 `.asmr-drama3-` 开头、以 `.part` 结尾的临时文件，可在确认传输已停止后手动清理。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import re
import tempfile
from pathlib import Path
from urllib.parse import quote

import requests

REPO = "fishaudio/drama-3"
REVISION = "78e5ab79e49aa9a25abe8799eff2c4d73a1c3216"
HUB = "https://huggingface.co"
DESTINATION = Path("/content/drive/MyDrive/AsmrHelper_Models/models/fish_drama3")
CHUNK = 8 * 1024 * 1024
TIMEOUT = (15, 120)


def parse_manifest(data):
    files = data["siblings"]
    if data["sha"] != REVISION or len(files) != 25:
        raise ValueError("公开模型的版本或文件数量与预期不符")
    manifest = {}
    for entry in files:
        name, size = entry["rfilename"], entry["size"]
        if not re.fullmatch(r"[A-Za-z0-9._-]+", name) or name in {".", ".."} or name in manifest:
            raise ValueError(f"不安全或重复的文件名: {name}")
        if type(size) is not int or size <= 0:
            raise ValueError(f"文件大小无效: {name}")
        lfs = entry.get("lfs")
        algorithm, expected = ("sha256", lfs["sha256"]) if lfs else ("sha1", entry["blobId"])
        if not re.fullmatch(r"[0-9a-f]+", expected) or len(expected) != (64 if lfs else 40):
            raise ValueError(f"文件哈希无效: {name}")
        if lfs and lfs["size"] != size:
            raise ValueError(f"LFS 文件大小与清单不符: {name}")
        manifest[name] = (size, algorithm, expected)
    return manifest


def verified(path, spec):
    size, algorithm, expected = spec
    if path.is_symlink() or not path.is_file() or path.stat().st_size != size:
        return False
    digest = hashlib.new(algorithm)
    if algorithm == "sha1":
        digest.update(f"blob {size}\0".encode("ascii"))
    with path.open("rb") as stream:
        while block := stream.read(CHUNK):
            digest.update(block)
    return digest.hexdigest() == expected


def transfer_file(session, destination, name, spec):
    target = destination / name
    if target.exists() or target.is_symlink():
        if not verified(target, spec):
            raise FileExistsError(f"同名文件与官方版本不同，请先检查: {target}")
        return "已校验，跳过"

    url = f"{HUB}/{REPO}/resolve/{REVISION}/{quote(name, safe='')}"
    temp = tempfile.NamedTemporaryFile(
        dir=destination, prefix=".asmr-drama3-", suffix=".part", delete=False
    )
    partial = Path(temp.name)
    try:
        with temp, session.get(url, stream=True, timeout=TIMEOUT) as response:
            response.raise_for_status()
            written = 0
            for block in response.iter_content(CHUNK):
                written += len(block)
                if written > spec[0]:
                    raise ValueError(f"下载超过官方文件大小: {name}")
                temp.write(block)
        if not verified(partial, spec):
            raise ValueError(f"文件大小或哈希校验失败: {name}")
        if target.exists() or target.is_symlink():
            if not verified(target, spec):
                raise FileExistsError(f"发布前发现同名文件冲突: {target}")
            return "已校验，跳过"
        partial.rename(target)
        return "已复制并校验"
    finally:
        partial.unlink(missing_ok=True)


def main():
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("请先运行上一格并挂载 Google Drive")
    with requests.Session() as session:
        manifest_url = f"{HUB}/api/models/{REPO}/revision/{REVISION}"
        response = session.get(manifest_url, params={"blobs": "true"}, timeout=TIMEOUT)
        response.raise_for_status()
        manifest = parse_manifest(response.json())
        DESTINATION.mkdir(parents=True, exist_ok=True)
        for number, (name, spec) in enumerate(manifest.items(), 1):
            print(f"[{number}/{len(manifest)}] {name}", flush=True)
            print(f"  {transfer_file(session, DESTINATION, name, spec)}", flush=True)
    print(f"完成，已确认全部 {len(manifest)} 个原版文件：{DESTINATION}")


main()